In [60]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [62]:
data = pd.read_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/ham_data.csv')

In [63]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [65]:
data[data["yesil_skor_notu"].isna()].sample(10)

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
50463,https://world.openfoodfacts.org/product/376017...,3.760179e+12,Chapelure extra – La Chanteracoise – 300 g,300.0,NaN,la chanteracoise,"Plant-based foods and beverages, ,, Plant-base...","Organic, ,, EU Organic, ,, Non-EU Agriculture,...",NaN,NaN,...,1.0,6.0,8.0,NaN,1.0,0.0,Chapelure extra,g,"[Plant-based foods and beverages, Plant-based ...","[Organic, EU Organic, Non-EU Agriculture, EU A..."
9044,https://world.openfoodfacts.org/product/594114...,5.941143e+12,deplina – paine – 300g,300.0,Plastic,paine,"Plant-based foods and beverages, ,, Plant-base...",NaN,NaN,NaN,...,0.0,0.0,0.0,4.0,0.0,0.0,deplina,g,"[Plant-based foods and beverages, Plant-based ...",[]
40174,https://world.openfoodfacts.org/product/325039...,3.250392e+12,Délice surimi & crabe – Intermarche – 120 g,120.0,NaN,intermarche,"Seafood, ,, Fishes and their products, ,, Fish...",NaN,NaN,NaN,...,0.0,1.0,8.0,NaN,0.0,0.0,Délice surimi & crabe,g,"[Seafood, Fishes and their products, Fish prep...",[]
26343,https://world.openfoodfacts.org/product/541466...,5.414661e+12,5414661000838,NaN,NaN,NaN,Dairies,"FSC, ,, Green Dot",Belgium,NaN,...,1.0,2.0,0.0,1.0,0.0,0.0,5414661000838,None,[Dairies],"[FSC, Green Dot]"
52505,https://world.openfoodfacts.org/product/611125...,6.111252e+12,Pepsi – 1L,1.0,NaN,pepsi,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,9.0,0.0,0.0,0.0,0.0,0.0,Pepsi,l,"[Beverages and beverages preparations, Beverag...",[]
24629,https://world.openfoodfacts.org/product/501104...,5.011040e+12,Cream Plain Flour – Odlums – 1 kg,1.0,Paper bag,odlums,"Plant-based foods and beverages, ,, Plant-base...",NaN,NaN,Ireland,...,0.0,0.0,2.0,3.0,1.0,0.0,Cream Plain Flour,kg,"[Plant-based foods and beverages, Plant-based ...",[]
9281,https://world.openfoodfacts.org/product/405648...,4.056490e+12,Mildly spiced red kidney beans in chilli sauce...,390.0,NaN,freshona,Beans,NaN,NaN,NaN,...,1.0,0.0,2.0,1.0,2.0,5.0,Mildly spiced red kidney beans in chilli sauce,g,[Beans],[]
8959,https://world.openfoodfacts.org/product/859400...,8.594001e+12,Magnesia plus – 0.7l,0.7,NaN,magnesia,"Beverages and beverages preparations, ,, Bever...",Vitamin B12 source,cs:Karlovy vary (Česká republika),NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,Magnesia plus,l,"[Beverages and beverages preparations, Beverag...",[Vitamin B12 source]
34552,https://world.openfoodfacts.org/product/520108...,5.201080e+12,Ζωμός Λαχανικών Knorr – 60 g,60.0,Paper,knorr,"Broths, ,, el:Ζωμός-λαχανικών, ,, el:Κύβοι-λαχ...",Sustainable farming,NaN,NaN,...,0.0,0.0,5.0,0.0,0.0,0.0,Ζωμός Λαχανικών Knorr,g,"[Broths, el:Ζωμός-λαχανικών, el:Κύβοι-λαχανικώ...",[Sustainable farming]
53210,https://world.openfoodfacts.org/product/611125...,6.111252e+12,Pepsi – 33cl,33.0,NaN,pepsi,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,9.0,0.0,20.0,0.0,0.0,0.0,Pepsi,cl,"[Beverages and beverages preparations, Beverag...",[]


In [66]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [67]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001546
urun_adi                         0.003093
miktar                          18.833027
ambalaj                         57.943492
markalar                         3.478032
kategoriler                      0.003093
etiketler                       30.055828
mensei                          75.496033
uretim_yerleri                  79.714829
satildigi_ulkeler                0.102068
icerik_metni                    12.195537
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   12.517205
nutriscore_notu                  0.091242
nova_grubu                      14.456490
yesil_skor_notu                 25.758162
palmiye_yagi_icermez            18.386094
vejetaryen                      22.184248
vegan_durumu                    12.006866
yag_seviyesi                     2.135688
doymus_yag_seviyesi              3.137807
seker_seviyesi                   2

In [68]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","etiketler","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)


In [69]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [70]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (64461): ['1.020304050607081e+17', '1000029852900.0', '10001219.0', '10001295.0', '10001356.0', '10001400.0', '10001404.0', '10001691.0', '10001707.0', '10001875.0'] ...
miktar (1143): ['0.0', '0.03', '0.042', '0.045', '0.046', '0.05', '0.055', '0.06', '0.065', '0.07'] ...
markalar (12450): ['"tradition culinaire"', "'z bregov", '(sans marque)', '07x netto 03.25', '1 2 3 fruits', '1 attimo in forma', '1 l', '1 x auer 01.25', '1%', '1-2-3'] ...
alerjenler (1836): ["['Acesulfame-potassium']", "['Apple', 'Banana', 'Celery', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Nuts']", "['Apple', 'Banana', 'Gluten', 'Sulphur dioxide and sulphites']", "['Apple', 'Banana', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Kiwi', 'Orange', 'Peach']", "['Apple', 'Banana', 'Milk']", "['Apple', 'Banana', 'Orange', 'Peach']", "['Apple', 'Banana', 'Orange', 'Sulphur dioxide and sulphites']"] ...
eser_miktarlar (2420): ["['07

In [71]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     39877
False    12897
Name: count, dtype: int64

In [72]:
data.isnull().mean() * 100

barkod                         0.001546
miktar                        18.833027
markalar                       3.478032
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 12.517205
nutriscore_notu                0.091242
nova_grubu                    14.456490
yesil_skor_notu               25.758162
palmiye_yagi_icermez          18.386094
vejetaryen                    22.184248
vegan_durumu                  12.006866
enerji_kcal                    0.947992
yag_g                          0.954178
doymus_yag_g                   1.998051
karbonhidrat_g                 1.036141
seker_g                        1.408843
lif_g                         29.135673
protein_g                      0.960364
tuz_g                          0.759321
alkol_yuzde                   95.046626
meyve_sebze_baklagil_yuzde    71.070009
birim                         20.784684
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [73]:
data.to_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/data.csv', index=False)

In [74]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'alkol_yuzde', 'meyve_sebze_baklagil_yuzde',
       'birim', 'kategori_listesi', 'etiketler_listesi'],
      dtype='object')